# Tutorial 09: Double pendulum I (cont.)


## 3. Quadratic programming with the Unitree Go2 quadruped robot



<img src="unitree-go2-quadruped.jpg" width="50%" align="" />

### Whole-body control

When formulated as a Quadratic Program (QP), Whole-Body Control (WBC), we solve an optimization problem that relates the joint-space to the task-space coordinates. This way, we avoid having to compute the inverse kinematics, which may be complex, have multiple solutions or even no analytical solution at all. 

Let's say we know how to calculate the task-space velocities from the joint-space velocities. Then, we can use the Jacobian to describe the relation between the two:

$$
V = J\dot{q}
$$

Where V are our task-space velocities and $\dot{q}$ are the joint-space velocities. We can solve the previous equation numerically using a minimization problem:

$$
\min_{\dot{q}} |J \dot{q} - \dot{V}|
$$

We can also express the above as a quadratic program:

$$
\min_{\dot{q}} \frac{1}{2} \dot{q}^T Q \dot{q} + c^T \dot{q}
$$

where $Q = J^T J$ and $c = -J^T V$.

We can do the same with the accelerations:

$$
\min_{\ddot{q}} |J \ddot{q} + \dot{J}\dot{q} - \dot{V}|
$$

And our resulting problem is:

$$
\min_{\ddot{q}} \frac{1}{2} \ddot{q}^T Q \ddot{q} + c^T \ddot{q}
$$

where $Q = J^T J$ and $c = -J^T (\dot{V} - \dot{J} \dot{q})$.

Consider the quadrupedal robot above. It has 12 DoF (3 per leg) and a floating base, i.e., a virtual 6 DoF linkage between the world frame and its base frame. We formulate the following quadratic program:

$
 \def\arraystretch{1.2}
 \begin{array}{cc}
\underset{\mathbf{x}}{\text{min}} & \frac{1}{2}\mathbf{x}^T\mathbf{Q}\mathbf{x} +\mathbf{c}^T\mathbf{x}\\
\text{s.t.} & \mathbf{A}\mathbf{x} = \mathbf{b}\\
 \end{array}
$

where $\mathbf{A}$ and $\mathbf{b}$ include the equations of motion and the rigid (non-moving) contact constraints for the 4 legs. For this task we use the classes Go2Sim (for simulation), RobotModel (for kinematics and dynamics), and the URDF of the robot ("urdf/go2.urdf"). 

1. Setup the above QP. When setting $\mathbf{Q}$ and $\mathbf{c}$ to zero, the solver will only output the gravity compensation torques, which hold the robot in its current position, as well as the respective contact forces. Send the computed torques to the robot.

The following code sets up the robot model using the pinocchio library:


In [1]:
from pydrake.all import StartMeshcat
import time
from qpsolvers import solve_qp
!pip install numpy==1.26.4
from robot_sim import Go2Sim, install_deepnote_nginx
from robot_model import RobotModel
from plotting import plot_all

install_deepnote_nginx()

# Start the meshcat visualizer here. You can click on the URL that is printed below to the robot visualization
meshcat = StartMeshcat()

# Load Robot model class, which uses pinocchio for kinematics and dynamics: https://github.com/stack-of-tasks/pinocchio
# Note: The order of joint positions (qin 19 x 1) in pinocchio will be as [floating base, legs] with:
# [x, y, z, qx, qy, qt, qw, 
#  bl_abad, bl_shoulder, bl_knee, br_abad, br_shoulder, br_knee, 
#  fl_abad, fl_shoulder, fl_knee, fr_abad, fr_shoulder, fr_knee]
#
# The order of joint velocities/acceleration (qd 18 x 1) in pinocchio will be as [floating base, legs] with:
# [vx, vy, vz, wx, wy, wz,
#  bl_abad, bl_shoulder, bl_knee, br_abad, br_shoulder, br_knee, 
#  fl_abad, fl_shoulder, fl_knee, fr_abad, fr_shoulder, fr_knee]
robot_model = RobotModel("/work/urdf/go2.urdf", floating_base=True)

Set up the Quadratic Program in the following cell:

- Add the constraints that enforce the equations of motion.

$$
M \ddot{q} + C + G = S\tau + J_c^T F
$$

Where $M$ is the mass matrix, $C+G$ is the bias forces term for the current configuration, $S$ is the selection matrix, $\tau$ are the joint torques, $J_c$ is the contact Jacobian, and $F$ is the contact forces.

In [2]:
import numpy as np

def setupQP(q,qd):
    jac_fr = robot_model.spaceJacobian(q, "fr_contact")
    jac_fl = robot_model.spaceJacobian(q, "fl_contact")
    jac_br = robot_model.spaceJacobian(q, "br_contact")
    jac_bl = robot_model.spaceJacobian(q, "bl_contact")
    base_jac = robot_model.spaceJacobian(q, "base_link")
    M = robot_model.massInertiaMatrix(q)
    tau_bias = robot_model.biasForces(q)
    X_base = robot_model.pose(q,"base_link")
    V_base = robot_model.twist(q, qd, "base_link")

    # Decision variables (42):
    #  - Joint Accelerations (18)
    #  - Joint Torques (12)
    #  - Contact Forces (4 x 3)
    nj = 18 + 12 + 12

    # Equality constraints: 
    #  - Equations of motion (18): M*qdd + C + G = S*tau + sum_j J_j^T F_j
    #  - Rigid contacts (4 x 3): J_j qdd = -dot{J}^T qd
    nc = 18 + 12     

    Q = np.identity(nj)*1e-8  # Optimization/Task matrix: For now empty, since we have no tasks (only add a small regularization term)
    c = np.zeros(nj)          # Gradient vector: For now empty, since we have no tasks
    A = np.zeros((nc,nj))     # Eq. constraint matrix
    b = np.zeros(nc)          # Eq. constraint vector
    C = d = None              # No inequality constraints
    lb = ub = None            # Optimization variables are unbounded


    ### Add your code here!
    # Set up A
    # A[,] = 

    # Set up b
    # b[] =

    ###

    return Q, c, C, d, A, b, lb, ub

Now, we run the Whole-Body Controller in a loop.
With this controller, each instant we retrieve the joint positions and velocities to calculate the Jacobians, mass matrix, bias forces, etc.

In [3]:
def runGo2Controller(duration):
    q_0 = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.126, 0.61, -1.22, -0.126, 0.61, -1.22, 0.126, 0.61, -1.22, -0.126, 0.61, -1.22]
    sim = Go2Sim(meshcat, urdf_file = "/work/urdf/go2.urdf",  q_0 = q_0, dt = 0.001)
    sim.position_hold(q_0[7:19], duration=1.0)

    t = 0
    print("Running Whole-Body Controller...")
    sim.startRecording()
    while t < duration:
        q = sim.get_positions()
        qd = sim.get_velocities()

        # Set up QP
        Q, c, C, d, A, b, lb, ub = setupQP(q,qd)

        # Solve QP
        solver_output = solve_qp(Q, c, C, d, A, b, lb, ub, verbose=True, solver="quadprog")

        acc = solver_output[0:18]
        tau = solver_output[18:30]
        F_ext = solver_output[30:42]

        sim.set_torques(tau)
        sim.step()
        time.sleep(0.001)
        t += 0.001

    sim.stopAndPublishRecording()

runGo2Controller(duration=5.0)    

### Think-Pair-Share (1 min)

- Why does the robot fall down at some point?

Now, add a controller on joint level $\dot{\mathbf{q}}_r = \mathbf{K}_p(\mathbf{q}_r-\mathbf{q})$ to stabilize the robot position and avoid falling down. 
You can do that by setting a cost function that favors accelerating in the direction which will reduce the error between desired and current positions. One option is be:

$$
\min_{\ddot{q}} \ddot{q}^2 - 10(q_r - q) \ddot{q} -3(\dot{q_r} - \dot{q}) \ddot{q}
$$

In [4]:
import numpy as np

def setupQP(q,qd):
    jac_fr = robot_model.spaceJacobian(q, "fr_contact")
    jac_fl = robot_model.spaceJacobian(q, "fl_contact")
    jac_br = robot_model.spaceJacobian(q, "br_contact")
    jac_bl = robot_model.spaceJacobian(q, "bl_contact")
    base_jac = robot_model.spaceJacobian(q, "base_link")
    M = robot_model.massInertiaMatrix(q)
    tau_bias = robot_model.biasForces(q)
    X_base = robot_model.pose(q,"base_link")
    V_base = robot_model.twist(q, qd, "base_link")

    # Decision variables (42):
    #  - Joint Accelerations (18)
    #  - Joint Torques (12)
    #  - Contact Forces (4 x 3)
    nj = 18 + 12 + 12

    # Equality constraints: 
    #  - Equations of motion (18): M*qdd + C + G = S*tau + sum_j J_j^T F_j
    #  - Rigid contacts (4 x 3): J_j qdd = -dot{J}^T qd
    nc = 18 + 12     

    Q = np.identity(nj)*1e-8  # Optimization/Task matrix: For now empty, since we have no tasks (only add a small regularization term)
    c = np.zeros(nj)          # Gradient vector: For now empty, since we have no tasks
    A = np.zeros((nc,nj))     # Eq. constraint matrix
    b = np.zeros(nc)          # Eq. constraint vector
    C = d = None              # No inequality constraints
    lb = ub = None            # Optimization variables are unbounded

    ### Add your code here!

    # Set up Q: Joint space tracking task: Task matrix is the identity matrix
    Q[6:18,6:18] = np.identity(12)

    # Set up c to move to a fixed joint position
    q_target = np.array([0.126, 0.61, -1.72, -0.126, 0.61, -1.22, 0.126, 0.61, -1.22, -0.126, 0.61, -1.22])
    
    # c[] = 

    # Set up A
    A[0:18,0:18]  = M
    A[6:18,18:30] = -np.identity(12) # Selection matrix
    A[0:18,30:33] = -np.transpose(jac_fr[0:3,:])
    A[0:18,33:36] = -np.transpose(jac_fl[0:3,:])
    A[0:18,36:39] = -np.transpose(jac_br[0:3,:])
    A[0:18,39:42] = -np.transpose(jac_bl[0:3,:])
    A[18:21,0:18] = jac_fr[0:3,:]
    A[21:24,0:18] = jac_fl[0:3,:]
    A[24:27,0:18] = jac_br[0:3,:]
    A[27:30,0:18] = jac_bl[0:3,:]

    # Set up b
    b[0:18] = -tau_bias
    ###

    return Q, c, C, d, A, b, lb, ub

runGo2Controller(duration=5.0)

Add a controller in Cartesian space ($\mathbf{V}=\mathbf{K}_p(\mathbf{X}_r-\mathbf{X})$) to control the robot's base motion. Move the z-position of the robot base in a sinosoidal trajectory. 

In [5]:
from scipy.spatial.transform import Rotation
t = 0.0
dt = 1e-3

def rot_diff(rot_a, rot_b):    
    # Compute relative rotation
    rot = np.array(rot_b)
    rot_mat = np.linalg.inv(rot).dot(rot_a)
    
    # Convert rotation matrix to axis-angle
    r = Rotation.from_matrix(rot_mat)
    rotvec = r.as_rotvec()  # This gives us axis * angle in one vector

    # Transform to correct coordinates
    diff = rot_a.dot(rotvec)
    
    return diff

def setupQP(q,qd):
    jac_fr = robot_model.spaceJacobian(q, "fr_contact")
    jac_fl = robot_model.spaceJacobian(q, "fl_contact")
    jac_br = robot_model.spaceJacobian(q, "br_contact")
    jac_bl = robot_model.spaceJacobian(q, "bl_contact")
    base_jac = robot_model.spaceJacobian(q, "base_link")
    M = robot_model.massInertiaMatrix(q)
    tau_bias = robot_model.biasForces(q)
    X_base = robot_model.pose(q,"base_link")
    V_base = robot_model.twist(q, qd, "base_link")

    # Decision variables (42):
    #  - Joint Accelerations (18)
    #  - Joint Torques (12)
    #  - Contact Forces (4 x 3)
    nj = 18 + 12 + 12

    # Equality constraints: 
    #  - Equations of motion (18): M*qdd + C + G = S*tau + sum_j J_j^T F_j
    #  - Rigid contacts (4 x 3): J_j qdd = -dot{J}^T qd
    nc = 18 + 12     

    Q = np.identity(nj)*1e-8  # Optimization/Task matrix: For now empty, since we have no tasks (only add a small regularization term)
    c = np.zeros(nj)          # Gradient vector: For now empty, since we have no tasks
    A = np.zeros((nc,nj))     # Eq. constraint matrix
    b = np.zeros(nc)          # Eq. constraint vector
    C = d = None              # No inequality constraints
    lb = ub = None            # Optimization variables are unbounded

    # Set up Q
    global t
    X_target = np.array([0.0,0.0,0.3+0.05*np.sin(t), 
                         0.0, 0.0, 0.0])
    X_base_vect = np.array([0.0]*6)
    X_base_vect[0:3] = X_base.translation
    X_base_vect[3:6] = -rot_diff(np.eye(3), X_base.rotation)
    V_base_vect = np.array([0.0]*6)
    V_base_vect[0:3] = V_base.linear
    V_base_vect[3:6] = V_base.angular
    V_r = 100*(X_target-X_base_vect) + 30*(-V_base_vect)
    ### Add your code here!
    
    # Finish setting up Q
    # Q [,] = 

    # Set up c
    # c[] = 

    # Set up A
    A[0:18,0:18]  = M
    A[6:18,18:30] = -np.identity(12) # Selection matrix
    A[0:18,30:33] = -np.transpose(jac_fr[0:3,:])
    A[0:18,33:36] = -np.transpose(jac_fl[0:3,:])
    A[0:18,36:39] = -np.transpose(jac_br[0:3,:])
    A[0:18,39:42] = -np.transpose(jac_bl[0:3,:])
    A[18:21,0:18] = jac_fr[0:3,:]
    A[21:24,0:18] = jac_fl[0:3,:]
    A[24:27,0:18] = jac_br[0:3,:]
    A[27:30,0:18] = jac_bl[0:3,:]

    # Set up b
    b[0:18] = -tau_bias

    ###

    t += dt

    return Q, c, C, d, A, b, lb, ub

runGo2Controller(duration=10.0)

## Think-Pair-Share (10 min)

Try other motions for the robot, such as rotating about the x, y, and z axes separately and then combine the motions.